In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")        
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv")

In [14]:
variances = X_train.var(axis=0)
print(" Variance distribution (on train set):")
print(f"    min    : {variances.min():.6f}")
print(f"    median : {variances.median():.6f}")
print(f"    mean   : {variances.mean():.6f}")
print(f"    max    : {variances.max():.6f}\n")

thresholds = [0.0, 0.001, 0.01]
threshold_labels = ["0.0 (exact zero only)", "0.001 (very low)", "0.01 (low)"]

for thresh, label in zip(thresholds, threshold_labels):
    n_removed = (variances <= thresh).sum()
    print(f"  Threshold {label:<23} → removes {n_removed:>3} features "
          f"({n_removed / len(variances) * 100:.1f}%)")

 Variance distribution (on train set):
    min    : 0.001665
    median : 0.070603
    mean   : 0.242389
    max    : 80.553185

  Threshold 0.0 (exact zero only)   → removes   0 features (0.0%)
  Threshold 0.001 (very low)        → removes   0 features (0.0%)
  Threshold 0.01 (low)              → removes  37 features (6.6%)


In [11]:
from sklearn.feature_selection import VarianceThreshold

VARIANCE_THRESHOLD = 0.01
selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
selector.fit(X_train)

removed_var_cols = X_train.columns[~selector.get_support()].tolist()
kept_cols        = X_train.columns[selector.get_support()].tolist()

X_train_clean = selector.transform(X_train)
X_test_clean  = selector.transform(X_test)

print(f"\n  Using threshold = {VARIANCE_THRESHOLD}")
print(f"  Features removed (near-zero var) : {len(removed_var_cols)}")
print(f"  Features kept                    : {len(kept_cols)}")
if removed_var_cols:
    print(f"  Removed examples                 : {removed_var_cols[:8]}")




  Using threshold = 0.01
  Features removed (near-zero var) : 37
  Features kept                    : 525
  Removed examples                 : ['tBodyAcc-mean()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-mean()-Z', 'tGravityAcc-std()-X', 'tGravityAcc-std()-Y', 'tGravityAcc-mad()-X', 'tGravityAcc-mad()-Y', 'tGravityAcc-iqr()-X']


In [24]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_clean)
X_test_final  = scaler.transform(X_test_clean)

print(f"Final feature matrix shape : {X_train_final.shape}")

Final feature matrix shape : (7352, 525)


In [33]:
from models import get_models, train_and_evaluate

models = get_models()
models

{'Decision Tree': DecisionTreeClassifier(random_state=42),
 'Random Forest': RandomForestClassifier(n_jobs=-1, random_state=42),
 'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Linear SVC': LinearSVC(max_iter=2000, random_state=42),
 'RBF SVM': SVC(C=10, random_state=42),
 'K-Nearest Neighbor': KNeighborsClassifier(n_jobs=-1)}

In [38]:
results , trained_models  = train_and_evaluate(models, X_train_final, y_train.values.ravel(), X_test_final, y_test.values.ravel())


Training Decision Tree...
Trained Decision Tree in 3.61 seconds.
Training Random Forest...
Trained Random Forest in 1.9 seconds.
Training Logistic Regression...
Trained Logistic Regression in 4.46 seconds.
Training Linear SVC...
Trained Linear SVC in 52.81 seconds.
Training RBF SVM...
Trained RBF SVM in 13.09 seconds.
Training K-Nearest Neighbor...
Trained K-Nearest Neighbor in 0.02 seconds.


In [37]:
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df

,Model,Precision,Recall,F1-Score,Accuracy,Training Time
2,Logistic Regression,0.96,0.96,0.96,0.96,3.16
3,Linear SVC,0.97,0.96,0.96,0.96,1.43
4,RBF SVM,0.95,0.95,0.95,0.95,0.93
1,Random Forest,0.93,0.93,0.93,0.93,1.47
0,Decision Tree,0.86,0.86,0.86,0.86,3.57
5,K-Nearest Neighbor,0.82,0.81,0.81,0.81,0.01


In [35]:
results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
results_df

,Model,Precision,Recall,F1-Score,Accuracy,Training Time
3,Linear SVC,0.96,0.96,0.96,0.96,8.02
4,RBF SVM,0.96,0.96,0.96,0.96,1.36
2,Logistic Regression,0.95,0.95,0.95,0.95,1.02
1,Random Forest,0.93,0.93,0.93,0.93,1.47
5,K-Nearest Neighbor,0.89,0.88,0.88,0.88,0.01
0,Decision Tree,0.86,0.86,0.86,0.86,3.57
